# Workshop 1 - Gold KPI package (solution)

Goal: build a BI-ready Gold object, define KPI, validate data quality and write
the decision log.

![Workshop success criteria](../assets/images/workshop_success_criteria.png)

In [ ]:
%run ../setup/00_setup

## Success criteria

You are done when:

1. `gold.fact_sales_dashboard` exists and has documented grain.
2. Revenue, Gross Margin and Return Rate are calculated from Gold.
3. Detail totals reconcile with the monthly aggregate.
4. At least three data-quality issues are named with example rows.
5. A view/table/aggregate decision is justified.
6. The self-check cell passes.

## Runtime pre-check

Run Module 2 before this workshop. The workshop starts from the BI-ready Gold
objects created there.

In [ ]:
required_tables = [
    f"{GOLD}.fact_sales_dashboard",
    f"{GOLD}.fact_sales_dashboard_monthly",
]

missing = [table for table in required_tables if not spark.catalog.tableExists(table)]
if missing:
    raise Exception("Missing tables. Run Module 2 first: " + ", ".join(missing))

print("[OK] Workshop inputs exist")

## Tasks

1. Build or inspect `gold.fact_sales_dashboard`.
2. Define Revenue, Gross Margin and Return Rate.
3. Find at least three data-quality issues.
4. Decide whether the reporting source should be a view, table or aggregate.
5. Fill the KPI dictionary and decision log templates.
6. Prepare a short BI contract paragraph for Power BI.

## Task 1 - inspect grain

In [ ]:
spark.sql(f"""
SELECT
  COUNT(*) AS rows,
  COUNT(DISTINCT line_id) AS distinct_line_ids,
  COUNT(*) - COUNT(DISTINCT line_id) AS duplicate_line_ids
FROM {GOLD}.fact_sales_dashboard
""").show()

## Task 2 - KPI definitions

In [ ]:
spark.sql(f"""
SELECT
  SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END) AS revenue,
  SUM(CASE WHEN is_completed THEN line_margin ELSE 0 END) AS gross_margin,
  COUNT(DISTINCT CASE WHEN is_returned THEN order_id END) AS returned_orders,
  COUNT(DISTINCT CASE WHEN is_completed THEN order_id END) AS completed_orders,
  ROUND(
    COUNT(DISTINCT CASE WHEN is_returned THEN order_id END)
    / NULLIF(
        COUNT(DISTINCT CASE WHEN is_completed THEN order_id END)
        + COUNT(DISTINCT CASE WHEN is_returned THEN order_id END),
        0
      ),
    4
  ) AS return_rate
FROM {GOLD}.fact_sales_dashboard
""").show()

## Task 3 - quality issues with evidence

In [ ]:
spark.sql(f"""
SELECT 'missing_price' AS issue, COUNT(*) AS rows FROM {GOLD}.fact_sales_dashboard WHERE unit_price IS NULL
UNION ALL SELECT 'invalid_status', COUNT(*) FROM {GOLD}.fact_sales_dashboard WHERE status NOT IN ('Completed','Cancelled','Returned')
UNION ALL SELECT 'future_order_date', COUNT(*) FROM {GOLD}.fact_sales_dashboard WHERE order_date > current_date()
""").show()

In [ ]:
spark.sql(f"""
SELECT line_id, order_id, order_date, status, category, unit_price, line_revenue
FROM {GOLD}.fact_sales_dashboard
WHERE unit_price IS NULL
   OR status NOT IN ('Completed','Cancelled','Returned')
   OR order_date > current_date()
ORDER BY order_date DESC
LIMIT 20
""").show(truncate=False)

## Task 4 - reconciliation

In [ ]:
spark.sql(f"""
WITH detail AS (
  SELECT ROUND(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 2) AS revenue
  FROM {GOLD}.fact_sales_dashboard
),
agg AS (
  SELECT ROUND(SUM(revenue), 2) AS revenue
  FROM {GOLD}.fact_sales_dashboard_monthly
)
SELECT detail.revenue AS detail_revenue, agg.revenue AS agg_revenue, detail.revenue - agg.revenue AS diff
FROM detail CROSS JOIN agg
""").show()

## Suggested decision

Use a table for `gold.fact_sales_dashboard` and an aggregate for summary report
pages. Avoid DirectQuery on Silver detail tables.

## Suggested BI contract paragraph

The executive dashboard uses `gold.fact_sales_dashboard_monthly` for summary
pages and `gold.fact_sales_dashboard` for detail validation. Grain is one row
per month, customer region, category and channel in the aggregate. Revenue and
gross margin include completed orders only. DirectQuery must not read Silver
detail tables.

## Self-check

In [ ]:
assert spark.catalog.tableExists(f"{GOLD}.fact_sales_dashboard"), "Missing dashboard fact"
assert spark.catalog.tableExists(f"{GOLD}.fact_sales_dashboard_monthly"), "Missing monthly aggregate"

grain = spark.sql(f"""
SELECT COUNT(*) AS rows, COUNT(DISTINCT line_id) AS distinct_line_ids
FROM {GOLD}.fact_sales_dashboard
""").first()
assert grain["rows"] >= grain["distinct_line_ids"], "Unexpected grain result"

recon = spark.sql(f"""
WITH detail AS (
  SELECT ROUND(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 2) AS revenue
  FROM {GOLD}.fact_sales_dashboard
),
agg AS (
  SELECT ROUND(SUM(revenue), 2) AS revenue
  FROM {GOLD}.fact_sales_dashboard_monthly
)
SELECT ABS(detail.revenue - agg.revenue) AS diff
FROM detail CROSS JOIN agg
""").first()["diff"]
assert recon < 0.01, f"Aggregate does not reconcile: {recon}"

print("[OK] Workshop 1 self-check passed")